In [1]:
import sys 

sys.path.append("..")

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import math
import numpy as np
from thop import clever_format

In [3]:
from dehazeddpm import MPRfusion, UNet, GaussianDiffusion
from dehazeddpm import make_beta_schedule

### Testing for the DehazeDDPM if it runs correctly

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Calculating on: {device}")

Calculating on: cuda


In [5]:
from ptflops import get_model_complexity_info

In [9]:
stage1_model = MPRfusion().to(device)
diffusion_timesteps = 1000

# Stage 2: Diffusion U-Net
# Ensure in_channel matches what you concatenate (e.g., J(3) + trmap(t) + Noisy(3) = 7d)
unet_model = UNet(
    in_channel = 7, 
    out_channel = 3, 
    inner_channel = 64, 
    attn_res=(16, ),
    res_blocks = 2,
    dropout = 0.2,
    image_size=256,
    with_noise_level_emb=True
).to(device)

s1_macs, s1_params = get_model_complexity_info(
    stage1_model, 
    (3, 256, 256),
    as_strings = False, 
    print_per_layer_stat = False,
    verbose=False
)

def unet_input_constructor(input_res):
    # 'input_res' is ignored here. We manually construct the exact tensors.
    batch_size = 1
    dummy_x = torch.randn(batch_size, 7, 256, 256, device=device)
    dummy_t = torch.randn(batch_size, device=device)

    # Return a dictionary that perfectly matches the arguments 
    # of your UNet's forward function: def forward(self, x, time):
    return {'x': dummy_x, 'time': dummy_t}
    

s2_macs, s2_params = get_model_complexity_info(
    unet_model, 
    (7, 256, 256),  # This placeholder is required but ignored by our constructor
    input_constructor=unet_input_constructor,
    as_strings = False, 
    print_per_layer_stat = False,
    verbose=False
)

# ==========================================
# calculate TOTAL Inference Cost
# ==========================================

total_params_raw = s1_params + s2_params 

# Total FLOPs = Stage1_FLOPs + (UNet_FLOPs * Timesteps)
total_macs_raw = s1_macs + (s2_macs * diffusion_timesteps)

# 5. Format for Printing
s1_macs_fmt, s1_params_fmt = clever_format([s1_macs, s1_params], "%.2f")
s2_macs_fmt, s2_params_fmt = clever_format([s2_macs, s2_params], "%.2f")
total_macs_fmt, total_params_fmt = clever_format([total_macs_raw, total_params_raw], "%.2f")

print("--- Individual Stage Complexity ---")
print(f"Stage 1: {s1_params_fmt} Params, {s1_macs_fmt} MACs")
print(f"Stage 2: {s2_params_fmt} Params, {s2_macs_fmt} MACs (per pass)")

print("\n==========================================")
print("TOTAL DEHAZEDDPM INFERENCE COMPLEXITY")
print(f"(Calculation: Stage1 + (Stage2 * {diffusion_timesteps} steps))")
print("==========================================")
print(f"Total Parameters: {total_params_fmt}")
print(f"Total Inference GFLOPs: {total_macs_fmt}")
print("==========================================")

Warning! No positional inputs found for a module, assuming batch size is 1.
--- Individual Stage Complexity ---
Stage 1: 2.78M Params, 10.96G MACs
Stage 2: 97.81M Params, 179.74G MACs (per pass)

TOTAL DEHAZEDDPM INFERENCE COMPLEXITY
(Calculation: Stage1 + (Stage2 * 1000 steps))
Total Parameters: 100.59M
Total Inference GFLOPs: 179.76T
